# PRIORIDADE 4: Tradução para o Município
### Conectando Políticas Públicas à Vida Real: Saúde, Educação, Previdência, Trabalho e Renda nos 5.571 Municípios Brasileiros

---

> 💡 **Como rodar este Notebook localmente no seu VS Code / Antigravity IDE:**
> 1. No canto superior direito deste notebook, clique no botão **"Select Kernel"** (ou no nome do kernel atual).
> 2. Selecione **`Python (trad_x_municipio)`** na lista de kernels disponíveis (ele já foi configurado e registrado no seu sistema com todas as bibliotecas necessárias: `pandas`, `openpyxl`, `numpy`, `jupyter`).
> 3. Se preferir selecionar pelo caminho do interpretador virtualenv, escolha:  
>    `/home/hianara/Voluntariado/venv_pmi/bin/python`

---

## 🎯 Objetivo da Prioridade 4
A **Prioridade 4** é a tarefa transversal e integradora de todo o projeto. Seu papel é transformar grandes números agregados nacionais em **narrativas tangíveis, humanas e municipalizadas** — respondendo diretamente à pergunta do cidadão: 
> *"O que tudo isso tem a ver com a minha cidade, meu trabalho e a minha família?"*

### 📋 As 4 Frentes Integradas:
1. **Saúde e Remédios (Prioridade 1):** Beneficiários da Farmácia Popular (hipertensão, diabetes, asma); pacientes de câncer (quimioterapia e radioterapia) e HIV; economia anual no bolso familiar com remédios gratuitos; e o impacto da falta de equipamentos locais e deslocamentos para polos regionais.
2. **Educação Pública (Prioridade 2):** Estudantes com Pé-de-Meia na poupança; livros didáticos gratuitos (PNLD); fomento a escolas em tempo integral; e o papel do Fundeb (quanto a União complementa e o risco de corte na merenda, manutenção e salários de professores).
3. **Previdência e Aposentados (Prioridade 3):** Aposentados e pensionistas do INSS (RGPS); valor anual injetado na economia local; a dependência do PIB municipal das aposentadorias; a proteção aos trabalhadores rurais; o risco de corte na renda das famílias; e o papel essencial do **BPC/LOAS** como rede de proteção social aos mais vulneráveis.
4. **Trabalho, CLT, MEI e Bolsa Família (Complementar):** Cidadãos com carteira assinada (CLT); Microempreendedores Individuais (MEI) ativos; total de empresas; famílias atendidas pelo Bolsa Família e valor mensal repassado.

---


In [1]:
import os
import re
import numpy as np
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# Configuração de formatação e exibição do Pandas
pd.set_option('display.max_columns', 70)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.'))

print("✅ Bibliotecas importadas com sucesso!")


✅ Bibliotecas importadas com sucesso!


## 📂 1. Carregamento das Bases de Dados
Carregamos as bases oficiais consolidadas das Prioridades 1, 2, 3 e a base de Trabalho e Renda:
- **Saúde:** `dados_prioridades/saude_municipios.xlsx` (DataSUS, Farmácia Popular, CEAF, Oncologia, HIV)
- **Educação:** `dados_prioridades/educacao_municipios_2025.xlsx` (MEC, FNDE, Censo Escolar 2024, Pé-de-Meia, Fundeb 2025)
- **Previdência:** `dados_prioridades/previdencia_municipios_2025.xlsx` (MPS, INSS, SIDRA/IBGE PIB Municipal 2023)
- **Trabalho e Renda:** `dados_prioridades/trabalho_renda/` (Bolsa Família, BPC, Vínculos CLT ativos, MEI ativos, empresas ativas)


In [2]:
PATH_SAUDE = 'dados_prioridades/saude_municipios.xlsx'
PATH_EDUC = 'dados_prioridades/educacao_municipios_2025.xlsx'
PATH_PREV = 'dados_prioridades/previdencia_municipios_2025.xlsx'
PATH_TRANSF = 'dados_prioridades/trabalho_renda/transferencias_renda_previdencia.csv'
PATH_EMP = 'dados_prioridades/trabalho_renda/empresas_emprego_servidores.csv'

# 1. Saúde
df_saude = pd.read_excel(PATH_SAUDE, sheet_name='Dados')
df_saude['cod_ibge7'] = df_saude['cod_ibge7'].astype(str).str.zfill(7)
df_saude['cod_ibge6'] = df_saude['cod_ibge6'].astype(str).str.zfill(6)

# 2. Educação
df_educ = pd.read_excel(PATH_EDUC, sheet_name='educacao_municipios_2025')
df_educ['cod_ibge_7'] = df_educ['cod_ibge_7'].astype(str).str.zfill(7)
df_educ['cod_ibge_6'] = df_educ['cod_ibge_6'].astype(str).str.zfill(6)

# 3. Previdência
df_prev = pd.read_excel(PATH_PREV, sheet_name='Dados')
df_prev['cod_ibge7'] = df_prev['cod_ibge7'].astype(str).str.zfill(7)
df_prev['cod_ibge6'] = df_prev['cod_ibge6'].astype(str).str.zfill(6)

# 4. Transferências e Renda (Bolsa Família e BPC)
df_transf = pd.read_csv(PATH_TRANSF, sep=';')
df_transf['cod_ibge7'] = df_transf['id_municipio_ibge'].astype(str).str.zfill(7)

# 5. Empresas, CLT e MEI
df_emp = pd.read_csv(PATH_EMP, sep=';')
df_emp['cod_ibge7'] = df_emp['id_municipio_ibge'].astype(str).str.zfill(7)

PATH_BEN = "ben_municipios_especie_2025.xlsx"
excel_ben = pd.ExcelFile(PATH_BEN)
sheet_ben = excel_ben.sheet_names[2]  # Valor_Médio_R$_dez24
df_ben_raw = pd.read_excel(PATH_BEN, sheet_name=sheet_ben, header=None)
df_ben = df_ben_raw.iloc[7:5578, [1, 3]].copy()
df_ben.columns = ["cod_ibge7", "valor_medio_aposentadoria_rs"]
df_ben["cod_ibge7"] = df_ben["cod_ibge7"].astype(str).str.zfill(7)
df_ben["valor_medio_aposentadoria_rs"] = pd.to_numeric(df_ben["valor_medio_aposentadoria_rs"], errors="coerce").round(2)

# Merge com df_prev
df_prev = df_prev.merge(df_ben, on="cod_ibge7", how="left")

# Importa módulo de cálculo estratégico da previdência
from calculo_impacto_previdencia import classificar_grupo, calcular_impacto_anual, obter_argumento_economico

df_prev["grupo_impacto_previdencia"] = df_prev["valor_medio_aposentadoria_rs"].apply(classificar_grupo)
df_prev["argumento_economico_previdencia"] = df_prev["valor_medio_aposentadoria_rs"].apply(obter_argumento_economico)

# Traz MEI e ME ativas para cálculo de impacto econômico no comércio local: perda / (MEI + ME)
df_prev = df_prev.merge(df_emp[['cod_ibge7', 'total_mei_ativos', 'total_me_ativas']], on='cod_ibge7', how='left')

def calc_prev_impact(r):
    mei = r.get("total_mei_ativos", 0)
    me = r.get("total_me_ativas", 0)
    total_pequenos_negocios = (mei + me) if (mei + me) > 0 else 1
    apos = r.get("aposentadorias_rgps_dezembro", 0)
    imp_ano, imp_com, imp_apos = calcular_impacto_anual(
        valor_liquido_apos_ano=r.get("valor_liquido_aposentadorias_ano_brl", 0),
        valor_medio=r.get("valor_medio_aposentadoria_rs", 0),
        comercios=total_pequenos_negocios,
        aposentados=apos
    )
    return pd.Series([imp_ano, imp_com, imp_apos], index=["perda_desvinculacao_anual_brl", "perda_desvinculacao_por_comercio_anual_brl", "perda_media_anual_por_aposentado_brl"])

impactos = df_prev.apply(calc_prev_impact, axis=1)
df_prev["perda_desvinculacao_anual_brl"] = impactos["perda_desvinculacao_anual_brl"]
df_prev["perda_desvinculacao_por_comercio_anual_brl"] = impactos["perda_desvinculacao_por_comercio_anual_brl"]
df_prev["perda_media_anual_por_aposentado_brl"] = impactos["perda_media_anual_por_aposentado_brl"]
df_prev["perda_hipotetica_anual_brl"] = df_prev["perda_desvinculacao_anual_brl"]
# Remove colunas temporárias para não conflitar no merge posterior
df_prev = df_prev.drop(columns=['total_mei_ativos', 'total_me_ativas'])

print(f"📊 Registros carregados com chaves cod_ibge7 e cod_ibge6:")
print(f" - Saúde: {len(df_saude)} municípios")
print(f" - Educação: {len(df_educ)} municípios")
print(f" - Previdência: {len(df_prev)} municípios")
print(f" - Transferências (Bolsa Família & BPC): {len(df_transf)} municípios")
print(f" - Trabalho & Empresas (CLT & MEI): {len(df_emp)} municípios")


📊 Registros carregados com chaves cod_ibge7 e cod_ibge6:
 - Saúde: 5571 municípios
 - Educação: 5571 municípios
 - Previdência: 5571 municípios
 - Transferências (Bolsa Família & BPC): 5571 municípios
 - Trabalho & Empresas (CLT & MEI): 5571 municípios


## 🔗 2. Unificação Cadastral dos 5.571 Municípios Brasileiros
Fazemos a junção pelas chaves de código IBGE de 7 dígitos (`cod_ibge7`), mantendo rigorosamente `cod_ibge7` e `cod_ibge6` como primeiras colunas de referência em todas as análises.


In [3]:
cols_saude = [
    'cod_ibge7', 'cod_ibge6', 'municipio', 'uf', 'no_regiao_saude', 'populacao_2025', 'municipio_pequeno',
    'fp_total', 'fp_so_hipertensao', 'fp_so_diabetes', 'fp_hipertensao_e_diabetes', 'fp_asma',
    'alto_custo_pacientes', 'quimioterapia_pacientes', 'radioterapia_pacientes', 'hiv_pessoas_em_tratamento_2022',
    'bolso_anual_minimo_rs', 'bolso_anual_mediano_rs',
    'radioterapia_pacientes_regiao_saude', 'quimioterapia_pacientes_regiao_saude',
    'txt_farmacia_popular', 'txt_bolso'
]

cols_educ = [
    'cod_ibge_7', 'regiao',
    'qtd_alunos_rede_publica', 'qtd_escolas_publicas', 'mat_em_publico',
    'qtd_beneficiarios_pe_de_meia', 'vl_pago_pe_de_meia', 'vl_por_aluno_pe_de_meia',
    'afetados_pnld', 'vl_pnld_total_est', 'vl_pnld_por_aluno_est',
    'vl_fundeb_rede_municipal', 'vl_fundeb_uniao_rede_municipal', 'pct_uniao_rede_municipal',
    'afetados_fundeb_complementacao', 'perda_fundeb_sem_uniao',
    'recebe_vaaf_rede_municipal', 'recebe_vaat_rede_municipal', 'recebe_vaar_rede_municipal',
    'afetados_eti', 'qtd_matriculas_eti_rede_municipal', 'vl_fomento_eti_rede_municipal',
    'txt_pe_de_meia', 'txt_fundeb_uniao', 'txt_eti', 'txt_afetados_pe_de_meia', 'txt_afetados_fundeb', 'txt_afetados_eti'
]

cols_prev = [
    'cod_ibge7', 'aposentadorias_rgps_dezembro', 'pensoes_por_morte_rgps_dezembro', 'total_rgps_dezembro',
    'valor_liquido_aposentadorias_ano_brl', 'valor_liquido_pensoes_ano_brl', 'valor_liquido_rgps_ano_brl',
    'beneficios_rgps_rurais_dezembro', 'valor_liquido_rgps_rural_ano_brl',
    'aposentadorias_por_100_habitantes', 'participacao_aposentadorias_no_rgps_pct', 'participacao_rural_no_rgps_pct',
    'pib_municipal_precos_correntes_brl', 'aposentadorias_sobre_pib_pct',
    'valor_medio_aposentadoria_rs', 'grupo_impacto_previdencia',
    'perda_desvinculacao_anual_brl', 'perda_desvinculacao_por_comercio_anual_brl', 'perda_media_anual_por_aposentado_brl',
    'perda_hipotetica_anual_brl', 'perda_hipotetica_por_habitante_brl',
    'argumento_economico_previdencia'
]

cols_transf = [
    'cod_ibge7', 'qtd_familias_bf', 'valor_total_bf_mes', 'ticket_medio_bf',
    'qtd_beneficiarios_bpc', 'valor_total_bpc_mes'
]

cols_emp = [
    'cod_ibge7', 'total_mei_ativos', 'total_me_ativas', 'total_me_comercio_ativas',
    'vinculos_clt_ativos', 'servidores_municipais_total', 'servidores_municipais_estatutarios',
    'servidores_municipais_comissionados', 'servidores_federais_siape', 'total_servidores_publicos'
]

# Junção Geral mantendo cod_ibge7 e cod_ibge6
df_mestre = df_saude[cols_saude].merge(df_educ[cols_educ], left_on='cod_ibge7', right_on='cod_ibge_7', how='left')
df_mestre = df_mestre.merge(df_prev[cols_prev], on='cod_ibge7', how='left')
df_mestre = df_mestre.merge(df_transf[cols_transf], on='cod_ibge7', how='left')
df_mestre = df_mestre.merge(df_emp[cols_emp], on='cod_ibge7', how='left')
df_mestre.drop(columns=['cod_ibge_7'], inplace=True, errors='ignore')

print(f"✅ Base Unificada construída com sucesso: {df_mestre.shape[0]} municípios e {df_mestre.shape[1]} colunas.")
print("Primeiras colunas:", df_mestre.columns[:6].tolist())


✅ Base Unificada construída com sucesso: 5571 municípios e 84 colunas.
Primeiras colunas: ['cod_ibge7', 'cod_ibge6', 'municipio', 'uf', 'no_regiao_saude', 'populacao_2025']


## 🗣️ 3. O Motor de Tradução: Histórias + Números (Com Tratamento de Dados Zerados)
Aqui construímos as narrativas personalizadas para cada município.
Criamos funções auxiliares que convertem valores monetários e contagens em formatos naturais da língua portuguesa.
Quando um indicador estiver zerado, ausente ou sem registro na base oficial, a informação é tratada com elegância e transparência:
> *"Não há informações oficiais sobre esse assunto."*


In [4]:
MSG_SEM_INFO = "Não há informações oficiais sobre esse assunto."

def fmt_num(val):
    if pd.isna(val) or val == 0:
        return "0"
    return f"{int(round(val)):,}".replace(",", ".")

def fmt_brl(val):
    if pd.isna(val) or val == 0:
        return "R$ 0"
    abs_v = abs(val)
    if abs_v >= 1e9:
        s = f"{abs_v / 1e9:.2f}".replace(".", ",")
        return f"R$ {s} bilhões"
    elif abs_v >= 1e6:
        s = f"{abs_v / 1e6:.1f}".replace(".", ",")
        return f"R$ {s} milhões"
    elif abs_v >= 1e3:
        s = f"{abs_v / 1e3:.1f}".replace(".", ",")
        return f"R$ {s} mil"
    else:
        return f"R$ {abs_v:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

def sanitize_texto(texto, prefixos_zerados=None):
    if not texto or pd.isna(texto):
        return MSG_SEM_INFO
    s = str(texto).strip()
    if s in ["0", "0.0", "R$ 0", "R$ 0,00", "R$ 0 mil", ""]:
        return MSG_SEM_INFO
    if prefixos_zerados:
        for p in prefixos_zerados:
            if s.startswith(p):
                return MSG_SEM_INFO
    return s

def gerar_texto_previdencia(row):
    apos = row.get("aposentadorias_rgps_dezembro", 0)
    vl_apos = row.get("valor_liquido_aposentadorias_ano_brl", 0)
    pensoes = row.get("pensoes_por_morte_rgps_dezembro", 0)
    vl_pensoes = row.get("valor_liquido_pensoes_ano_brl", 0)
    pct_pib = row.get("aposentadorias_sobre_pib_pct", np.nan)
    pct_rural = row.get("participacao_rural_no_rgps_pct", np.nan)
    vl_med = row.get("valor_medio_aposentadoria_rs", 0)
    imp_ano = row.get("perda_desvinculacao_anual_brl", 0)
    imp_com = row.get("perda_desvinculacao_por_comercio_anual_brl", 0)
    imp_apos = row.get("perda_media_anual_por_aposentado_brl", 0)
    grupo = row.get("grupo_impacto_previdencia", "")
    mun = row["municipio"]

    if pd.isna(apos) or apos == 0:
        return MSG_SEM_INFO

    partes = []
    vl_med_fmt = f"R$ {vl_med:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".") if pd.notna(vl_med) and vl_med > 0 else ""
    partes.append(f"Em {mun}, {fmt_num(apos)} moradores recebem aposentadoria pelo INSS (benefício médio de {vl_med_fmt}), injetando {fmt_brl(vl_apos)} por ano direto na economia local.")
    
    if pd.notna(pensoes) and pensoes > 0:
        partes.append(f"Outras {fmt_num(pensoes)} famílias contam com pensão por morte ({fmt_brl(vl_pensoes)} anuais).")
        
    if pd.notna(pct_pib) and pct_pib > 0:
        pct_pib_fmt = f"{pct_pib:.1f}".replace(".", ",")
        partes.append(f"Esse dinheiro é vital para a cidade: o valor das aposentadorias equivale a {pct_pib_fmt}% de toda a riqueza produzida no município (PIB).")
        
    if pd.notna(pct_rural) and pct_rural >= 40:
        pct_rural_fmt = f"{pct_rural:.1f}".replace(".", ",")
        partes.append(f"A economia rural é especialmente protegida: {pct_rural_fmt}% dos recursos do INSS vão para famílias do campo e agricultura familiar.")
        
    if pd.notna(imp_ano) and imp_ano > 0:
        imp_com_fmt = f" ({fmt_brl(imp_com)} a menos por comércio/pequeno negócio ao ano)" if pd.notna(imp_com) and imp_com > 0 else ""
        if "Grupo 1" in str(grupo):
            partes.append(
                f"Com benefício médio no piso de 1 salário mínimo, o congelamento do Salário Mínimo (fim do ganho real acima da inflação) "
                f"retiraria {fmt_brl(imp_ano)} por ano do comércio local{imp_com_fmt}, "
                f"somado a uma perda média de {fmt_brl(imp_apos)} ao ano no bolso de cada aposentado e ao risco de corte no remédio gratuito da Farmácia Popular."
            )
        else:
            partes.append(
                f"Com benefício médio acima de 1 salário mínimo, uma eventual desindexação dos benefícios "
                f"+ revogação da isenção do Imposto de Renda até R$ 5 mil drenaria {fmt_brl(imp_ano)} anuais da economia municipal "
                f"{imp_com_fmt}, somada a uma perda média de {fmt_brl(imp_apos)} ao ano por aposentado e ao fim do remédio gratuito na Farmácia Popular."
            )
    return " ".join(partes)

def gerar_texto_bolsa_familia(row):
    fam = row.get('qtd_familias_bf', 0)
    vl_mes = row.get('valor_total_bf_mes', 0)
    ticket = row.get('ticket_medio_bf', 0)
    mun = row['municipio']
    
    if pd.isna(fam) or fam == 0:
        return MSG_SEM_INFO
    return (f"Em {mun}, {fmt_num(fam)} famílias recebem o Bolsa Família para garantir comida no prato e necessidades básicas, "
            f"com repasse total de {fmt_brl(vl_mes)} por mês à cidade (benefício médio de {fmt_brl(ticket)} por família).")

def gerar_texto_bpc(row):
    bpc = row.get('qtd_beneficiarios_bpc', 0)
    vl_bpc = row.get('valor_total_bpc_mes', 0)
    mun = row['municipio']
    
    if pd.isna(bpc) or bpc == 0:
        return MSG_SEM_INFO
    return (f"O BPC/LOAS garante a sobrevivência de {fmt_num(bpc)} idosos vulneráveis e pessoas com deficiência em {mun}, "
            f"injetando {fmt_brl(vl_bpc)} todos os meses na subsistência das famílias e no comércio de bairro.")

def gerar_texto_trabalho(row):
    clt = row.get('vinculos_clt_ativos', 0)
    mei = row.get('total_mei_ativos', 0)
    emp = row.get('total_empresas_ativas', 0)
    me_tot = row.get('total_me_ativas', 0)
    me_com = row.get('total_me_comercio_ativas', 0)
    serv_mun = row.get('servidores_municipais_total', 0)
    serv_fed = row.get('servidores_federais_siape', 0)
    total_serv = row.get('total_servidores_publicos', 0)
    mun = row['municipio']
    
    partes = []
    if pd.notna(clt) and clt > 0:
        partes.append(f"{fmt_num(clt)} trabalhadores têm emprego com carteira assinada (CLT)")
    if pd.notna(mei) and mei > 0:
        partes.append(f"{fmt_num(mei)} empreendedores atuam como MEI (Microempreendedor Individual)")
    if pd.notna(me_tot) and me_tot > 0:
        if pd.notna(me_com) and me_com > 0:
            partes.append(f"a cidade reúne {fmt_num(me_tot)} microempresas ativas ({fmt_num(me_com)} dedicadas ao comércio)")
        else:
            partes.append(f"a cidade reúne {fmt_num(me_tot)} microempresas ativas")
    if pd.notna(total_serv) and total_serv > 0:
        partes.append(f"{fmt_num(total_serv)} são servidores públicos ({fmt_num(serv_mun)} municipais e {fmt_num(serv_fed)} federais)")
    # empresas ativas removido a pedido do usuario
        
    if partes:
        if len(partes) == 1:
            return f"No mercado de trabalho em {mun}, {partes[0]}."
        return f"No mercado de trabalho em {mun}, {', '.join(partes[:-1])} e {partes[-1]}."
    return MSG_SEM_INFO

def gerar_texto_saude_equipamentos(row):
    quimio = row.get('quimioterapia_pacientes', 0)
    radio = row.get('radioterapia_pacientes', 0)
    quimio_reg = row.get('quimioterapia_pacientes_regiao_saude', 0)
    radio_reg = row.get('radioterapia_pacientes_regiao_saude', 0)
    reg_nome = row.get('no_regiao_saude', 'região de saúde')
    mun = row['municipio']
    pequeno = row.get('municipio_pequeno', False)
    
    if pequeno:
        if (pd.notna(quimio_reg) and quimio_reg > 0) or (pd.notna(radio_reg) and radio_reg > 0):
            return (f"Por ser um município de pequeno porte, {mun} não dispõe de equipamentos de alta complexidade (como aceleradores lineares para radioterapia). "
                    f"Os pacientes dependem da {reg_nome}, onde {fmt_num(quimio_reg)} realizam quimioterapia e {fmt_num(radio_reg)} fazem radioterapia.")
        return MSG_SEM_INFO
    else:
        textos = []
        if pd.notna(quimio) and quimio > 0:
            textos.append(f"{fmt_num(quimio)} pacientes fazem tratamento quimioterápico contra o câncer")
        if pd.notna(radio) and radio > 0:
            verbo_radio = "realiza" if radio == 1 else "realizam"
            textos.append(f"{fmt_num(radio)} {verbo_radio} radioterapia no SUS")
        if textos:
            return f"Na assistência oncológica em {mun}, {' e '.join(textos)}, com retaguarda da {reg_nome}."
        return MSG_SEM_INFO

def gerar_narrativa_consolidada(row):
    mun = row['municipio']
    uf = row['uf']
    pop = row['populacao_2025']
    
    # 1. Saúde
    txt_fp = row.get('txt_farmacia_popular', '')
    txt_bolso = row.get('txt_bolso', '')
    alto_custo = row.get('alto_custo_pacientes', 0)
    
    tem_fp = pd.notna(txt_fp) and str(txt_fp).strip() != MSG_SEM_INFO and not str(txt_fp).startswith('0 moradores')
    tem_bolso = pd.notna(txt_bolso) and str(txt_bolso).strip() != MSG_SEM_INFO and not str(txt_bolso).startswith('Sem o programa, gastariam pelo menos R$ 0')
    
    if tem_fp and tem_bolso:
        saude_txt = f"• SAÚDE: {txt_fp}. {txt_bolso}."
    elif tem_fp:
        saude_txt = f"• SAÚDE: {txt_fp}."
    else:
        saude_txt = "• SAÚDE: Não há informações oficiais sobre atendimentos da Farmácia Popular no município."
        
    if pd.notna(alto_custo) and alto_custo > 0:
        if tem_fp:
            saude_txt += f" Além disso, {fmt_num(alto_custo)} pacientes recebem remédios de alto custo pelo SUS (para doenças raras e graves)."
        else:
            saude_txt += f" Por outro lado, {fmt_num(alto_custo)} pacientes recebem remédios de alto custo pelo SUS (para doenças raras e graves)."
    elif not tem_fp:
        saude_txt = f"• SAÚDE: {MSG_SEM_INFO}"
        
    # 2. Educação
    txt_pe = row.get('txt_pe_de_meia', '')
    txt_fund = row.get('txt_fundeb_uniao', '')
    txt_afet_fund = row.get('txt_afetados_fundeb', '')
    escolas = row.get('qtd_escolas_publicas', 0)
    alunos = row.get('qtd_alunos_rede_publica', 0)
    
    tem_pe = pd.notna(txt_pe) and str(txt_pe).strip() != MSG_SEM_INFO and not str(txt_pe).startswith('0 estudantes')
    tem_fund = pd.notna(txt_fund) and str(txt_fund).strip() != MSG_SEM_INFO and not str(txt_fund).startswith('R$ 0')
    tem_afet = pd.notna(txt_afet_fund) and str(txt_afet_fund).strip() != MSG_SEM_INFO and not str(txt_afet_fund).startswith('0 alunos')
    
    partes_educ = []
    if tem_pe:
        partes_educ.append(txt_pe)
    if tem_fund and pd.notna(escolas) and escolas > 0:
        partes_educ.append(f"Para manter {fmt_num(escolas)} escolas e {fmt_num(alunos)} alunos, {txt_fund}")
    elif tem_fund:
        partes_educ.append(txt_fund)
    if tem_afet:
        partes_educ.append(txt_afet_fund)
        
    if partes_educ:
        educ_txt = "• EDUCAÇÃO: " + ". ".join(partes_educ) + "."
    else:
        educ_txt = f"• EDUCAÇÃO: {MSG_SEM_INFO}"
        
    # 3. Trabalho & Renda
    fam_bf = row.get('qtd_familias_bf', 0)
    vl_bf = row.get('valor_total_bf_mes', 0)
    clt = row.get('vinculos_clt_ativos', 0)
    mei = row.get('total_mei_ativos', 0)
    total_serv = row.get('total_servidores_publicos', 0)
    bpc = row.get('qtd_beneficiarios_bpc', 0)
    
    frases_renda = []
    trab_p = []
    if pd.notna(clt) and clt > 0:
        trab_p.append(f"{fmt_num(clt)} cidadãos trabalham com carteira assinada (CLT)")
    if pd.notna(mei) and mei > 0:
        trab_p.append(f"{fmt_num(mei)} são microempreendedores (MEI)")
    if pd.notna(total_serv) and total_serv > 0:
        trab_p.append(f"{fmt_num(total_serv)} atuam como servidores públicos")
        
    if trab_p:
        if len(trab_p) == 1:
            frases_renda.append(trab_p[0])
        else:
            frases_renda.append(", ".join(trab_p[:-1]) + " e " + trab_p[-1])
            
    soc_p = []
    if pd.notna(fam_bf) and fam_bf > 0:
        soc_p.append(f"{fmt_num(fam_bf)} famílias contam com o Bolsa Família ({fmt_brl(vl_bf)}/mês injetados no comércio local)")
    if pd.notna(bpc) and bpc > 0:
        soc_p.append(f"{fmt_num(bpc)} pessoas vulneráveis e com deficiência recebem o BPC/LOAS")
        
    if soc_p:
        frases_renda.append("Na rede de proteção social, " + " e ".join(soc_p))
        
    if frases_renda:
        renda_txt = "• TRABALHO & RENDA: " + ". ".join(frases_renda) + "."
    else:
        renda_txt = f"• TRABALHO & RENDA: {MSG_SEM_INFO}"
        
    # 4. Previdência & Economia
    apos = row.get('aposentadorias_rgps_dezembro', 0)
    vl_apos = row.get('valor_liquido_aposentadorias_ano_brl', 0)
    pct_pib = row.get('aposentadorias_sobre_pib_pct', np.nan)
    perda_ano = row.get('perda_hipotetica_anual_brl', 0)
    
    vl_med = row.get("valor_medio_aposentadoria_rs", 0)
    imp_ano = row.get("perda_desvinculacao_anual_brl", 0)
    grupo = row.get("grupo_impacto_previdencia", "")
    if pd.notna(apos) and apos > 0:
        vl_med_fmt = f" (média de R$ {vl_med:,.2f})".replace(",", "X").replace(".", ",").replace("X", ".") if pd.notna(vl_med) and vl_med > 0 else ""
        prev_txt = f"• PREVIDÊNCIA & ECONOMIA: {fmt_num(apos)} moradores recebem aposentadoria pelo INSS{vl_med_fmt}, garantindo {fmt_brl(vl_apos)} injetados na economia da cidade todos os anos."
        if pd.notna(pct_pib) and pct_pib > 0:
            pct_pib_fmt = f"{pct_pib:.1f}".replace(".", ",")
            prev_txt += f" Esse valor equivale a {pct_pib_fmt}% do PIB municipal — sustentando mercados, feiras e farmácias locais."
        imp_com = row.get("perda_desvinculacao_por_comercio_anual_brl", 0)
        if pd.notna(imp_ano) and imp_ano > 0:
            imp_com_fmt = f" ({fmt_brl(imp_com)} a menos por comércio/pequeno negócio da cidade a cada ano)" if pd.notna(imp_com) and imp_com > 0 else ""
            if "Grupo 1" in str(grupo):
                prev_txt += f" A desvinculação e o congelamento do Salário Mínimo (fim do ganho real) drenariam {fmt_brl(imp_ano)} ao ano do comércio local{imp_com_fmt}, somado ao risco sobre a Farmácia Popular."
            else:
                prev_txt += f" A desindexação dos benefícios e a revogação da isenção do IR até R$ 5 mil retirariam {fmt_brl(imp_ano)} anuais da economia municipal{imp_com_fmt}, somada ao fim da Farmácia Popular."
    else:
        prev_txt = f"• PREVIDÊNCIA & ECONOMIA: {MSG_SEM_INFO}"
        
    narrativa = (
        f"Em {mun} ({uf}), com {fmt_num(pop)} habitantes, as políticas públicas federais sustentam diretamente o cotidiano da população:\n\n"
        f"{saude_txt}\n\n"
        f"{educ_txt}\n\n"
        f"{renda_txt}\n\n"
        f"{prev_txt}"
    )
    return narrativa

print("⏳ Tratando colunas de textos originais e gerando narrativas traduzidas...")

# Sanitização das colunas originais caso estejam zeradas
df_mestre['txt_farmacia_popular'] = df_mestre['txt_farmacia_popular'].apply(
    lambda x: sanitize_texto(x, ['0 moradores', '0 pessoas'])
)
df_mestre['txt_bolso'] = df_mestre['txt_bolso'].apply(
    lambda x: sanitize_texto(x, ['Sem o programa, gastariam pelo menos R$ 0'])
)
df_mestre['txt_pe_de_meia'] = df_mestre['txt_pe_de_meia'].apply(
    lambda x: sanitize_texto(x, ['0 estudantes'])
)
df_mestre['txt_fundeb_uniao'] = df_mestre['txt_fundeb_uniao'].apply(
    lambda x: sanitize_texto(x, ['R$ 0'])
)
df_mestre['txt_afetados_fundeb'] = df_mestre['txt_afetados_fundeb'].apply(
    lambda x: sanitize_texto(x, ['0 alunos'])
)
df_mestre['txt_eti'] = df_mestre['txt_eti'].apply(
    lambda x: sanitize_texto(x, ['0 matrículas', '0 alunos'])
)
df_mestre['txt_afetados_pe_de_meia'] = df_mestre['txt_afetados_pe_de_meia'].apply(
    lambda x: sanitize_texto(x, ['Se o Pé-de-Meia for interrompido, 0'])
)
df_mestre['txt_afetados_eti'] = df_mestre['txt_afetados_eti'].apply(
    lambda x: sanitize_texto(x, ['0 alunos'])
)

# Geração das novas colunas traduzidas
df_mestre['txt_prev_completo'] = df_mestre.apply(gerar_texto_previdencia, axis=1)
df_mestre['txt_bolsa_familia'] = df_mestre.apply(gerar_texto_bolsa_familia, axis=1)
df_mestre['txt_bpc'] = df_mestre.apply(gerar_texto_bpc, axis=1)
df_mestre['txt_trabalho_clt_mei'] = df_mestre.apply(gerar_texto_trabalho, axis=1)
df_mestre['txt_saude_equipamentos'] = df_mestre.apply(gerar_texto_saude_equipamentos, axis=1)
df_mestre['narrativa_consolidada'] = df_mestre.apply(gerar_narrativa_consolidada, axis=1)
print("✅ Narrativas e textos tratados com sucesso para todos os 5.571 municípios!")


⏳ Tratando colunas de textos originais e gerando narrativas traduzidas...


✅ Narrativas e textos tratados com sucesso para todos os 5.571 municípios!


## 📖 4. Glossário do Cidadão: Traduzindo o 'Tecniquês' para a Vida Real
Muitas vezes, a população e as lideranças locais não percebem a importância das políticas públicas porque os nomes são siglas frias e burocráticas. Abaixo está o glossário que traduz os termos técnicos para o linguajar popular:


In [5]:
glossario_dados = [
    {
        "Sigla / Termo": "Bolsa Família",
        "Nome Oficial": "Programa Bolsa Família",
        "O que é na prática": "Programa de transferência direta de renda com condicionalidades de saúde (vacinação) e educação (frequência escolar), garantindo um valor mínimo de R$ 600 por família mais adicionais para crianças e gestantes.",
        "Por que importa para a cidade": "O dinheiro cai no cartão e é 100% gasto no comércio da cidade (arroz, feijão, botijão de gás, feira livre). É o que sustenta as vendas do pequeno varejo nas cidades mais pobres.",
        "Atenção / Risco": "Exige manter o Cadastro Único atualizado e os filhos com frequência escolar mínima de 85%."
    },
    {
        "Sigla / Termo": "BPC / LOAS",
        "Nome Oficial": "Benefício de Prestação Continuada da Lei Orgânica da Assistência Social",
        "O que é na prática": "É um salário mínimo mensal garantido pela Constituição para idosos com 65 anos ou mais e pessoas com deficiência de qualquer idade que vivem em famílias de extrema pobreza (renda menor que 1/4 do salário mínimo por pessoa).",
        "Por que importa para a cidade": "Ao contrário da aposentadoria, não exige ter contribuído com o INSS. É a garantia de que pessoas com deficiência e idosos vulneráveis não passem fome. Esse dinheiro vai direto para comida e remédio no comércio local.",
        "Atenção / Risco": "O BPC NÃO paga 13º salário e NÃO deixa pensão por morte. Exige cadastro atualizado no CadÚnico a cada 2 anos."
    },
    {
        "Sigla / Termo": "CLT (Carteira Assinada)",
        "Nome Oficial": "Vínculos Empregatícios Formais (Consolidação das Leis do Trabalho)",
        "O que é na prática": "O trabalhador contratado com registro em carteira de trabalho, garantindo salário fixo, FGTS, férias remuneradas, 13º salário, seguro-desemprego e proteção previdenciária.",
        "Por que importa para a cidade": "Mede o grau de formalização e solidez econômica do município. Cidades com mais empregos CLT têm maior arrecadação e maior poder de compra estável.",
        "Atenção / Risco": "A informalidade excessiva fragiliza os trabalhadores e sobrecarrega a assistência social quando ocorrem crises econômicas."
    },
    {
        "Sigla / Termo": "MEI",
        "Nome Oficial": "Microempreendedor Individual",
        "O que é na prática": "Modelo simplificado de formalização para quem trabalha por conta própria (faturamento até R$ 81 mil/ano). Paga um valor fixo mensal reduzido de imposto e tem CNPJ e acesso a direitos do INSS.",
        "Por que importa para a cidade": "Representa a força do empreendedorismo local: pequenos prestadores de serviços, eletricistas, cabeleireiros, mecânicos e comerciantes que movimentam os bairros.",
        "Atenção / Risco": "Inadimplência na guia DAS-MEI pode suspender a cobertura previdenciária (como auxílio-doença ou salário-maternidade)."
    },
    {
        "Sigla / Termo": "Fundeb",
        "Nome Oficial": "Fundo de Manutenção e Desenvolvimento da Educação Básica",
        "O que é na prática": "É o cofre central que financia as creches, escolas fundamentais e ensino médio. Reúne impostos locais e recebe um reforço bilionário de dinheiro da União.",
        "Por que importa para a cidade": "Sem o Fundeb, a maioria das prefeituras não conseguiria pagar o piso dos professores nem reformar escolas. Pelo menos 70% desse dinheiro tem que ir obrigatoriamente para salários de profissionais da educação.",
        "Atenção / Risco": "O risco não é o fundo acabar, mas cortes ou desvios na complementação da União atingirem diretamente os municípios menores que não arrecadam quase nada sozinhos."
    },
    {
        "Sigla / Termo": "VAAF, VAAT e VAAR",
        "Nome Oficial": "Modalidades de Complementação da União ao Fundeb",
        "O que é na prática": "São as 3 formas pelas quais o governo federal coloca dinheiro a mais nas escolas: VAAF (socorre redes com menor arrecadação geral), VAAT (socorre especificamente os municípios mais pobres) e VAAR (bônus por bater metas de gestão e equidade).",
        "Por que importa para a cidade": "O VAAT é a salvação de prefeituras com poucos recursos próprios. E o VAAR é um dinheiro extra que premia boas gestões escolares.",
        "Atenção / Risco": "Se o município não cumprir as condicionalidades do VAAR (como escolha técnica de diretores), perde essa fatia de recursos."
    },
    {
        "Sigla / Termo": "Pé-de-Meia",
        "Nome Oficial": "Programa Pé-de-Meia (Poupança do Ensino Médio)",
        "O que é na prática": "Um incentivo financeiro pago direto na conta poupança do estudante do ensino médio público cuja família está no CadÚnico. Paga R$ 200 por mês por frequência, mais R$ 1.000 ao passar de ano e R$ 200 ao fazer o Enem.",
        "Por que importa para a cidade": "Evita que os jovens abandonem a escola para fazer 'bicos'. O dinheiro cai no celular do jovem e circula na padaria, mercado e lojas da cidade.",
        "Atenção / Risco": "Depende da prefeitura e do estado manterem o envio rigoroso da frequência escolar todo mês ao MEC."
    },
    {
        "Sigla / Termo": "PNLD",
        "Nome Oficial": "Programa Nacional do Livro e do Material Didático",
        "O que é na prática": "O Ministério da Educação compra em escala nacional e entrega de graça os livros didáticos, literários e materiais pedagógicos em cada escola pública.",
        "Por que importa para a cidade": "As famílias não gastam um centavo com livros escolares, o que pesaria centenas de reais por filho no início do ano letivo.",
        "Atenção / Risco": "Qualquer corte federal na compra de livros desabastece as salas de aula, pois nenhuma prefeitura tem orçamento para comprar livros do próprio bolso."
    },
    {
        "Sigla / Termo": "Farmácia Popular",
        "Nome Oficial": "Programa Farmácia Popular do Brasil",
        "O que é na prática": "Parceria do SUS com farmácias particulares onde o cidadão retira de graça remédios essenciais para hipertensão (pressão alta), diabetes, asma, osteoporose e anticoncepcionais.",
        "Por que importa para a cidade": "A pessoa não precisa enfrentar fila de posto de saúde quando falta remédio na rede municipal. Ela vai na drogaria da esquina com a receita e o CPF e pega o remédio pago pela União.",
        "Atenção / Risco": "Se o programa for desfinanciado, o custo cai 100% no bolso das famílias (mais de R$ 100 a R$ 300 por mês em medicamentos) ou sobrecarrega os postos de saúde."
    },
    {
        "Sigla / Termo": "CEAF / Alto Custo",
        "Nome Oficial": "Componente Especializado da Assistência Farmacêutica",
        "O que é na prática": "Fornecimento de remédios de altíssimo custo para doenças graves e crônicas (como câncer, hepatites, esclerose múltipla, artrite reumatoide e doenças raras).",
        "Por que importa para a cidade": "São caixas de remédios que custam de R$ 3 mil a mais de R$ 50 mil por mês — valores impagáveis até para famílias de classe média.",
        "Atenção / Risco": "Atrasos em repasses federais ou compra centralizada deixam pacientes graves em risco iminente de morte."
    },
    {
        "Sigla / Termo": "RGPS (Previdência)",
        "Nome Oficial": "Regime Geral de Previdência Social (INSS)",
        "O que é na prática": "O sistema que paga as aposentadorias, pensões por morte, auxílio-doença e salário-maternidade aos trabalhadores da iniciativa privada, autônomos e trabalhadores rurais.",
        "Por que importa para a cidade": "Em mais de 70% dos municípios brasileiros, o volume de dinheiro pago pelo INSS aos aposentados é maior do que o FPM (Fundo de Participação dos Municípios) e sustenta a economia do interior.",
        "Atenção / Risco": "Qualquer proposta de desvinculação da aposentadoria do salário mínimo ou cortes no RGPS drena a circulação de dinheiro no comércio local."
    }
]

df_glossario = pd.DataFrame(glossario_dados)
df_glossario


,Sigla / Termo,Nome Oficial,O que é na prática,Por que importa para a cidade,Atenção / Risco
0,Bolsa Família,Programa Bolsa Família,Programa de transferência direta de renda com ...,O dinheiro cai no cartão e é 100% gasto no com...,Exige manter o Cadastro Único atualizado e os ...
1,BPC / LOAS,Benefício de Prestação Continuada da Lei Orgân...,É um salário mínimo mensal garantido pela Cons...,"Ao contrário da aposentadoria, não exige ter c...",O BPC NÃO paga 13º salário e NÃO deixa pensão ...
2,CLT (Carteira Assinada),Vínculos Empregatícios Formais (Consolidação d...,O trabalhador contratado com registro em carte...,Mede o grau de formalização e solidez econômic...,A informalidade excessiva fragiliza os trabalh...
3,MEI,Microempreendedor Individual,Modelo simplificado de formalização para quem ...,Representa a força do empreendedorismo local: ...,Inadimplência na guia DAS-MEI pode suspender a...
4,Fundeb,Fundo de Manutenção e Desenvolvimento da Educa...,"É o cofre central que financia as creches, esc...","Sem o Fundeb, a maioria das prefeituras não co...","O risco não é o fundo acabar, mas cortes ou de..."
5,"VAAF, VAAT e VAAR",Modalidades de Complementação da União ao Fundeb,São as 3 formas pelas quais o governo federal ...,O VAAT é a salvação de prefeituras com poucos ...,Se o município não cumprir as condicionalidade...
6,Pé-de-Meia,Programa Pé-de-Meia (Poupança do Ensino Médio),Um incentivo financeiro pago direto na conta p...,Evita que os jovens abandonem a escola para fa...,Depende da prefeitura e do estado manterem o e...
7,PNLD,Programa Nacional do Livro e do Material Didático,O Ministério da Educação compra em escala naci...,As famílias não gastam um centavo com livros e...,Qualquer corte federal na compra de livros des...
8,Farmácia Popular,Programa Farmácia Popular do Brasil,Parceria do SUS com farmácias particulares ond...,A pessoa não precisa enfrentar fila de posto d...,"Se o programa for desfinanciado, o custo cai 1..."
9,CEAF / Alto Custo,Componente Especializado da Assistência Farmac...,Fornecimento de remédios de altíssimo custo pa...,São caixas de remédios que custam de R$ 3 mil ...,Atrasos em repasses federais ou compra central...


## 🔍 5. Ferramenta de Consulta Rápida por Município
Crie relatórios executivos instantâneos para qualquer um dos 5.571 municípios do país digitando o nome da cidade ou o código IBGE (com 7 ou 6 dígitos):


In [6]:
def consultar_municipio(termo_busca):
    termo = str(termo_busca).strip().lower()
    
    if termo.isdigit():
        resultado = df_mestre[df_mestre['cod_ibge7'].str.contains(termo) | df_mestre['cod_ibge6'].astype(str).str.contains(termo)]
    else:
        resultado = df_mestre[df_mestre['municipio'].str.lower().str.contains(termo, na=False)]
        
    if resultado.empty:
        print(f"❌ Nenhum município encontrado para a busca: '{termo_busca}'")
        return None
        
    row = resultado.iloc[0]
    
    print("=" * 95)
    print(f"🏛️ BOLETIM DE TRADUÇÃO MUNICIPAL: {row['municipio'].upper()} ({row['uf']})")
    print(f"📍 Código IBGE 7 dígitos: {row['cod_ibge7']} | Código IBGE 6 dígitos: {row['cod_ibge6']}")
    print(f"👥 População Estimada (2025): {fmt_num(row['populacao_2025'])} habitantes | Região de Saúde: {row['no_regiao_saude']}")
    print("=" * 95)
    print("")
    print("📜 NARRATIVA CONSOLIDADA (PRONTA PARA USO):")
    print("")
    print(row['narrativa_consolidada'])
    print("")
    print("-" * 95)
    print("🔬 DETALHAMENTO POR ÁREA:")
    print("")
    print(f"💊 [SAÚDE] Farmácia Popular: {row['txt_farmacia_popular']}")
    print(f"   Economia no Bolso: {row['txt_bolso']}")
    print(f"   Equipamentos / Alta Complexidade: {row['txt_saude_equipamentos']}")
    print("")
    print(f"🎒 [EDUCAÇÃO] Pé-de-Meia: {row['txt_pe_de_meia']}")
    print(f"   Fundeb União: {row['txt_fundeb_uniao']}")
    print(f"   Alunos Protegidos pelo Fundeb: {row['txt_afetados_fundeb']}")
    print("")
    print(f"💼 [TRABALHO & EMPREGO]:")
    print(f"   {row['txt_trabalho_clt_mei']}")
    print(f"   {row['txt_bolsa_familia']}")
    print(f"   {row['txt_bpc']}")
    print("")
    print(f"👴 [PREVIDÊNCIA & ECONOMIA]:")
    print(f"   {row['txt_prev_completo']}")
    print("=" * 95)
    return row

print("Exemplo: Consulta pelo município de São Miguel do Guaporé (RO)")
consultar_municipio("São Miguel do Guaporé")


Exemplo: Consulta pelo município de São Miguel do Guaporé (RO)
🏛️ BOLETIM DE TRADUÇÃO MUNICIPAL: SÃO MIGUEL DO GUAPORÉ (RO)
📍 Código IBGE 7 dígitos: 1100320 | Código IBGE 6 dígitos: 110032
👥 População Estimada (2025): 22.305 habitantes | Região de Saúde: Central

📜 NARRATIVA CONSOLIDADA (PRONTA PARA USO):

Em São Miguel do Guaporé (RO), com 22.305 habitantes, as políticas públicas federais sustentam diretamente o cotidiano da população:

• SAÚDE: Não há informações oficiais sobre atendimentos da Farmácia Popular no município. Por outro lado, 164 pacientes recebem remédios de alto custo pelo SUS (para doenças raras e graves).

• EDUCAÇÃO: 390 estudantes receberam o Pé-de-Meia em São Miguel do Guaporé em 2025. Para manter 15 escolas e 4.686 alunos, R$ 744,3 MIL do Fundeb da rede municipal vieram da União em 2025. 4.483 alunos de São Miguel do Guaporé estudam em redes que recebem complementação da União ao Fundeb.

• TRABALHO & RENDA: 3.515 cidadãos trabalham com carteira assinada (CLT), 

cod_ibge7                                                                               1100320
cod_ibge6                                                                                110032
municipio                                                                 São Miguel do Guaporé
uf                                                                                           RO
no_regiao_saude                                                                         Central
populacao_2025                                                                            22305
municipio_pequeno                                                                         False
fp_total                                                                                      0
fp_so_hipertensao                                                                             0
fp_so_diabetes                                                                                0
fp_hipertensao_e_diabetes               

## 📊 6. Análises e Rankings de Destaque Nacional
Identificamos os municípios onde as políticas públicas são mais determinantes para enriquecer análises comparativas e reportagens:


In [7]:
top_clt = df_mestre.sort_values(by='vinculos_clt_ativos', ascending=False)[
    ['cod_ibge7', 'cod_ibge6', 'municipio', 'uf', 'populacao_2025', 'vinculos_clt_ativos', 'total_mei_ativos', 'qtd_familias_bf']
].head(5)

print("🏆 Top 5 Municípios com maior número de trabalhadores CLT com carteira assinada:")
display(top_clt)

top_mei = df_mestre.sort_values(by='total_mei_ativos', ascending=False)[
    ['cod_ibge7', 'cod_ibge6', 'municipio', 'uf', 'populacao_2025', 'total_mei_ativos', 'vinculos_clt_ativos']
].head(5)

print("\n🏆 Top 5 Municípios com maior número de Microempreendedores Individuais (MEI):")
display(top_mei)

top_bf = df_mestre.sort_values(by='qtd_familias_bf', ascending=False)[
    ['cod_ibge7', 'cod_ibge6', 'municipio', 'uf', 'populacao_2025', 'qtd_familias_bf', 'valor_total_bf_mes', 'ticket_medio_bf']
].head(5)

print("\n🏆 Top 5 Municípios com maior número de famílias no Bolsa Família:")
display(top_bf)


🏆 Top 5 Municípios com maior número de trabalhadores CLT com carteira assinada:


,cod_ibge7,cod_ibge6,municipio,uf,populacao_2025,vinculos_clt_ativos,total_mei_ativos,qtd_familias_bf
3829,3550308,355030,São Paulo,SP,11904961,4955653,1379115,636158
3242,3304557,330455,Rio de Janeiro,RJ,6730729,2030390,762878,520194
5570,5300108,530010,Brasília,DF,2996899,1070795,276043,86026
2309,3106200,310620,Belo Horizonte,MG,2415872,1023842,296252,110812
4005,4106902,410690,Curitiba,PR,1830795,815311,227175,74632



🏆 Top 5 Municípios com maior número de Microempreendedores Individuais (MEI):


,cod_ibge7,cod_ibge6,municipio,uf,populacao_2025,total_mei_ativos,vinculos_clt_ativos
3829,3550308,355030,São Paulo,SP,11904961,1379115,4955653
3242,3304557,330455,Rio de Janeiro,RJ,6730729,762878,2030390
2309,3106200,310620,Belo Horizonte,MG,2415872,296252,1023842
5570,5300108,530010,Brasília,DF,2996899,276043,1070795
2162,2927408,292740,Salvador,BA,2564204,272631,685043



🏆 Top 5 Municípios com maior número de famílias no Bolsa Família:


,cod_ibge7,cod_ibge6,municipio,uf,populacao_2025,qtd_familias_bf,valor_total_bf_mes,ticket_medio_bf
3829,3550308,355030,São Paulo,SP,11904961,636158,"450.845.174,60","708,70"
3242,3304557,330455,Rio de Janeiro,RJ,6730729,520194,"369.961.972,80","711,20"
949,2304400,230440,Fortaleza,CE,2578483,322854,"224.157.532,20","694,30"
2162,2927408,292740,Salvador,BA,2564204,286316,"201.108.358,40","702,40"
111,1302603,130260,Manaus,AM,2303732,268420,"185.210.000,00","690,00"


## 📑 7. Exportação para Planilha Excel Consolidada (`traducao_municipios_consolidado.xlsx`)
Criamos uma planilha executiva completa com 8 abas, incluindo **`cod_ibge7`** e **`cod_ibge6`** em todas as abas de dados, uma aba de **`Trabalho_e_Renda`** (CLT, MEI e Bolsa Família) e uma aba de **`Dicionario_de_Dados`** detalhada:
1. `Narrativas_Consolidadas`: Textos prontos para uso em discursos, redes sociais e folhetos.
2. `Saude_Medicamentos`: Farmácia Popular, câncer, HIV e economia no bolso.
3. `Educacao_Escolas`: Pé-de-Meia, livros do PNLD, matrículas e complementação da União ao Fundeb.
4. `Previdencia_Economia`: Aposentadorias, pensões, peso no PIB e simulação de corte de 10%.
5. `Trabalho_e_Renda`: Vínculos CLT ativos, MEI ativos, empresas ativas, famílias no Bolsa Família e BPC.
6. `Dicionario_de_Dados`: Descrição estruturada de cada coluna, tipo, unidade, fonte e referência.
7. `Glossario_Cidadao`: O dicionário com a tradução dos termos técnicos.
8. `Metodologia_Fontes`: Detalhamento das fontes e parâmetros oficiais.


In [8]:
OUTPUT_EXCEL = 'traducao_municipios_consolidado.xlsx'
print(f"⏳ Gravando planilha Excel estruturada: {OUTPUT_EXCEL}...")

# 1. Narrativas Consolidadas (com cod_ibge7 e cod_ibge6)
cols_aba_narrativas = [
    'cod_ibge7', 'cod_ibge6', 'municipio', 'uf', 'regiao', 'populacao_2025',
    'narrativa_consolidada',
    'txt_farmacia_popular', 'txt_bolso', 'txt_saude_equipamentos',
    'txt_pe_de_meia', 'txt_fundeb_uniao', 'txt_afetados_fundeb',
    'txt_trabalho_clt_mei', 'txt_bolsa_familia', 'txt_bpc',
    'txt_prev_completo'
]
df_aba_narrativas = df_mestre[cols_aba_narrativas].rename(columns={
    'cod_ibge7': 'cod_ibge7',
    'cod_ibge6': 'cod_ibge6',
    'municipio': 'municipio',
    'uf': 'uf',
    'regiao': 'regiao',
    'populacao_2025': 'populacao_2025',
    'narrativa_consolidada': 'historia_consolidada_municipio',
    'txt_farmacia_popular': 'txt_farmacia_popular',
    'txt_bolso': 'txt_bolso',
    'txt_saude_equipamentos': 'txt_saude_equipamentos',
    'txt_pe_de_meia': 'txt_pe_de_meia',
    'txt_fundeb_uniao': 'txt_fundeb_uniao',
    'txt_afetados_fundeb': 'txt_afetados_fundeb',
    'txt_trabalho_clt_mei': 'txt_trabalho_clt_mei',
    'txt_bolsa_familia': 'txt_bolsa_familia',
    'txt_bpc': 'txt_bpc',
    'txt_prev_completo': 'txt_prev_completo'
})

# 2. Saúde (com cod_ibge7 e cod_ibge6)
cols_aba_saude = [
    'cod_ibge7', 'cod_ibge6', 'municipio', 'uf', 'no_regiao_saude', 'populacao_2025', 'municipio_pequeno',
    'fp_total', 'fp_so_hipertensao', 'fp_so_diabetes', 'fp_hipertensao_e_diabetes', 'fp_asma',
    'alto_custo_pacientes', 'quimioterapia_pacientes', 'radioterapia_pacientes', 'hiv_pessoas_em_tratamento_2022',
    'bolso_anual_minimo_rs', 'bolso_anual_mediano_rs',
    'quimioterapia_pacientes_regiao_saude', 'radioterapia_pacientes_regiao_saude'
]
df_aba_saude = df_mestre[cols_aba_saude]

# 3. Educação (com cod_ibge7 e cod_ibge6)
cols_aba_educ = [
    'cod_ibge7', 'cod_ibge6', 'municipio', 'uf', 'regiao', 'populacao_2025',
    'qtd_alunos_rede_publica', 'qtd_escolas_publicas', 'mat_em_publico',
    'qtd_beneficiarios_pe_de_meia', 'vl_pago_pe_de_meia', 'vl_por_aluno_pe_de_meia',
    'afetados_pnld', 'vl_pnld_total_est',
    'vl_fundeb_rede_municipal', 'vl_fundeb_uniao_rede_municipal', 'pct_uniao_rede_municipal',
    'afetados_fundeb_complementacao', 'perda_fundeb_sem_uniao',
    'recebe_vaaf_rede_municipal', 'recebe_vaat_rede_municipal', 'recebe_vaar_rede_municipal',
    'afetados_eti', 'qtd_matriculas_eti_rede_municipal', 'vl_fomento_eti_rede_municipal'
]
df_aba_educ = df_mestre[cols_aba_educ]

# 4. Previdência (com cod_ibge7 e cod_ibge6)
cols_aba_prev = [
    'cod_ibge7', 'cod_ibge6', 'municipio', 'uf', 'populacao_2025',
    'aposentadorias_rgps_dezembro', 'pensoes_por_morte_rgps_dezembro', 'total_rgps_dezembro',
    'valor_liquido_aposentadorias_ano_brl', 'valor_liquido_pensoes_ano_brl', 'valor_liquido_rgps_ano_brl',
    'valor_medio_aposentadoria_rs', 'grupo_impacto_previdencia',
    'pib_municipal_precos_correntes_brl', 'aposentadorias_sobre_pib_pct',
    'beneficios_rgps_rurais_dezembro', 'participacao_rural_no_rgps_pct',
    'perda_desvinculacao_anual_brl', 'perda_desvinculacao_por_comercio_anual_brl', 'perda_media_anual_por_aposentado_brl',
    'argumento_economico_previdencia'
]
df_aba_prev = df_mestre[cols_aba_prev]

# 5. Trabalho e Renda (Bolsa Família, CLT, MEI, BPC)
cols_aba_trabalho = [
    'cod_ibge7', 'cod_ibge6', 'municipio', 'uf', 'populacao_2025',
    'vinculos_clt_ativos', 'total_mei_ativos', 'total_me_ativas', 'total_me_comercio_ativas',
    'qtd_familias_bf', 'valor_total_bf_mes', 'ticket_medio_bf',
    'qtd_beneficiarios_bpc', 'valor_total_bpc_mes',
    'servidores_municipais_total', 'servidores_municipais_estatutarios', 'servidores_municipais_comissionados',
    'servidores_federais_siape', 'total_servidores_publicos'
]
df_aba_trabalho = df_mestre[cols_aba_trabalho].rename(columns={
    'cod_ibge7': 'cod_ibge7',
    'cod_ibge6': 'cod_ibge6',
    'municipio': 'municipio',
    'uf': 'uf',
    'populacao_2025': 'populacao_2025',
    'vinculos_clt_ativos': 'trabalhadores_clt_ativos',
    'total_mei_ativos': 'total_mei_ativos',
    'total_me_ativas': 'total_microempresas_ativas',
    'total_me_comercio_ativas': 'total_microempresas_comercio_ativas',
    'qtd_familias_bf': 'familias_bolsa_familia',
    'valor_total_bf_mes': 'valor_mensal_bolsa_familia_rs',
    'ticket_medio_bf': 'ticket_medio_bolsa_familia_rs',
    'qtd_beneficiarios_bpc': 'beneficiarios_bpc_loas',
    'valor_total_bpc_mes': 'valor_mensal_bpc_rs',
    'servidores_municipais_total': 'servidores_municipais',
    'servidores_municipais_estatutarios': 'servidores_municipais_estatutarios',
    'servidores_municipais_comissionados': 'servidores_municipais_comissionados',
    'servidores_federais_siape': 'servidores_federais_siape',
    'total_servidores_publicos': 'total_servidores_publicos'
})

# 6. Dicionário de Dados Completo - Enriquecido com Fórmulas, Cálculos e Explicações
df_dic = pd.read_excel("traducao_municipios_consolidado.xlsx", sheet_name="Dicionario_de_Dados")

# Remove registros obsoletos
df_dic = df_dic[~df_dic["Coluna"].isin(["total_empresas_ativas", "perda_hipotetica_anual_brl", "perda_hipotetica_por_habitante_brl", "perda_desvinculacao_por_habitante_brl"])]

# Atualiza e enriquece descrições e fórmulas
atualizacoes_dic = {
    "perda_desvinculacao_anual_brl": {
        "Descrição": "Impacto financeiro anual estimado na economia da cidade caso haja desvinculação / fim do ganho real do Salário Mínimo. Cálculo: valor_liquido_aposentadorias_ano_brl * 0,0722 (taxa de 7,22% obtida pela soma da mediana da inflação IPCA de 4,57% com a mediana do PIB positivo de 2,65% no período 2016-2025).",
        "Fonte": "Cenário Analítico (MPS/INSS Dez/2025 + IBGE)",
        "Tipo": "Simulação",
        "Unidade": "R$/ano"
    },
        "perda_media_anual_por_aposentado_brl": {
        "Descrição": "Perda anual média estimada por morador aposentado pelo INSS. Cálculo: perda_desvinculacao_anual_brl / aposentadorias_rgps_dezembro. Representa quanto cada aposentado da cidade deixa de receber ao ano devido à desvinculação ou congelamento do ganho real.",
        "Fonte": "Cenário Analítico (MPS/INSS Dez/2025)",
        "Tipo": "Simulação",
        "Unidade": "R$/aposentado/ano",
        "Período / referência": "2025/2026",
        "Tabela / Aba": "Previdencia_Economia"
    },
    "perda_desvinculacao_por_comercio_anual_brl": {
        "Descrição": "Perda anual média estimada por pequeno negócio e comércio local. Cálculo: perda_desvinculacao_anual_brl / (total_mei_ativos + total_microempresas_ativas). Mede quanto cada comerciante/MEI da cidade deixa de faturar no ano devido à redução da massa de rendimento dos aposentados.",
        "Fonte": "Cenário Analítico (MPS/INSS + Mapa de Empresas + Simples)",
        "Tipo": "Simulação",
        "Unidade": "R$/comércio/ano",
        "Período / referência": "2025/2026",
        "Tabela / Aba": "Previdencia_Economia"
    },
    "valor_medio_aposentadoria_rs": {
        "Descrição": "Valor líquido médio dos benefícios de aposentadoria pagos no município pelo RGPS em Dezembro/2025.",
        "Fonte": "MPS / INSS (Síntese Dezembro - Tabela Total de Aposentadorias)",
        "Tipo": "Dado",
        "Unidade": "R$/benefício",
        "Período / referência": "12/2025"
    },
    "grupo_impacto_previdencia": {
        "Descrição": "Classificação de vulnerabilidade: Grupo 1 se valor_medio_aposentadoria_rs <= R$ 1.518,00 (1 Salário Mínimo); Grupo 2 se valor_medio_aposentadoria_rs > R$ 1.518,00.",
        "Fonte": "Critério Metodológico Estratégico",
        "Tipo": "Classificação",
        "Unidade": "Texto",
        "Período / referência": "2025/2026"
    },
    "argumento_economico_previdencia": {
        "Descrição": "Linha argumentativa personalizada: Grupo 1 foca no congelamento do piso SM + corte na Farmácia Popular; Grupo 2 foca na desindexação + revogação da isenção do IR até R$ 5 mil + corte na Farmácia Popular.",
        "Fonte": "Matriz Estratégica",
        "Tipo": "Texto",
        "Unidade": "Texto",
        "Período / referência": "2025/2026"
    },
    "aposentadorias_sobre_pib_pct": {
        "Descrição": "Peso relativo das aposentadorias na economia municipal. Cálculo: (valor_liquido_aposentadorias_ano_brl / pib_municipal_precos_correntes_brl) * 100."
    },
    "participacao_rural_no_rgps_pct": {
        "Descrição": "Participação da previdência rural na seguridade do município. Cálculo: (beneficios_rgps_rurais_dezembro / total_rgps_dezembro) * 100."
    },
    "pct_uniao_rede_municipal": {
        "Descrição": "Fatia do orçamento do Fundeb municipal financiada pela União. Cálculo: (vl_fundeb_uniao_rede_municipal / vl_fundeb_rede_municipal) * 100."
    },
    "vl_por_aluno_pe_de_meia": {
        "Descrição": "Valor médio anual do incentivo financeiro por estudante atendido. Cálculo: vl_pago_pe_de_meia / qtd_beneficiarios_pe_de_meia."
    },
    "ticket_medio_bolsa_familia_rs": {
        "Descrição": "Benefício financeiro médio por família atendida no Bolsa Família. Cálculo: valor_mensal_bolsa_familia_rs / familias_bolsa_familia."
    },
    "total_servidores_publicos": {
        "Descrição": "Total de servidores públicos em exercício no município. Cálculo: servidores_municipais + servidores_federais_siape."
    },
    "bolso_anual_minimo_rs": {
        "Descrição": "Economia anual mínima que os pacientes têm com remédios gratuitos de pressão, diabetes e asma. Cálculo baseado nos preços mínimos de genéricos regulados pela Anvisa/CMED."
    },
    "bolso_anual_mediano_rs": {
        "Descrição": "Economia anual mediana proporcionada pela Farmácia Popular às famílias, calculada com base no preço de balcão mediano praticado em farmácias (Anvisa/CMED)."
    },
    "trabalhadores_clt_ativos": {
        "Descrição": "Estoque de empregos formais ativos com carteira assinada (CLT) no município em Agosto de 2026.",
        "Fonte": "MTE / Novo CAGED (Tabela 8.1 - série com ajustes)",
        "Período / referência": "08/2026"
    }
}

for col_name, dict_info in atualizacoes_dic.items():
    if col_name in df_dic["Coluna"].values:
        for k, v in dict_info.items():
            df_dic.loc[df_dic["Coluna"] == col_name, k] = v
    else:
        nova_linha = {
            "Tabela / Aba": dict_info.get("Tabela / Aba", "Geral"),
            "Coluna": col_name,
            "Tipo": dict_info.get("Tipo", "Calculado"),
            "Unidade": dict_info.get("Unidade", "—"),
            "Descrição": dict_info.get("Descrição", ""),
            "Fonte": dict_info.get("Fonte", "Oficial"),
            "Período / referência": dict_info.get("Período / referência", "2025/2026")
        }
        df_dic = pd.concat([df_dic, pd.DataFrame([nova_linha])], ignore_index=True)

# Adiciona regiao se não constar
if "regiao" not in df_dic["Coluna"].values:
    nova_reg = {
        "Tabela / Aba": "Todas",
        "Coluna": "regiao",
        "Tipo": "Categórico",
        "Unidade": "Texto",
        "Descrição": "Macrorregião geográfica oficial do município (Norte, Nordeste, Centro-Oeste, Sudeste, Sul).",
        "Fonte": "IBGE",
        "Período / referência": "2025"
    }
    df_dic = pd.concat([df_dic, pd.DataFrame([nova_reg])], ignore_index=True)

df_dicionario_completo = df_dic

# 7. Glossário Cidadão
glossario_cidadao = [
    {"Sigla / Termo": "Farmácia Popular", "Nome Oficial": "Programa Farmácia Popular do Brasil (PFPB)", "O que é na prática": "Distribuição 100% gratuita na rede credenciada de remédios contínuos para pressão alta, diabetes e asma.", "Por que importa para a cidade": "Mantém idosos e doentes crônicos fora do hospital e desonera o SUS municipal.", "Atenção / Risco": "Sem o programa, as famílias mais pobres teriam que tirar do orçamento de comida para comprar remédio."},
    {"Sigla / Termo": "Pé-de-Meia", "Nome Oficial": "Programa Pé-de-Meia (Poupança do Ensino Médio)", "O que é na prática": "Incentivo mensal de R$ 200 + poupança de R$ 1.000 ao fim de cada ano letivo concluído para jovens do CadÚnico.", "Por que importa para a cidade": "Reduz drasticamente o abandono escolar e injeta recursos que os jovens gastam no comércio local.", "Atenção / Risco": "O corte do benefício empurraria jovens vulneráveis para o subemprego ou informalidade precoce."},
    {"Sigla / Termo": "Fundeb da União", "Nome Oficial": "Complementação da União ao FUNDEB (VAAF, VAAT, VAAR)", "O que é na prática": "Aporte financeiro direto do governo federal para redes de ensino municipais que não conseguem atingir o investimento mínimo por aluno.", "Por que importa para a cidade": "Garante pagamento do piso salarial dos professores, merenda, reformas e transporte escolar.", "Atenção / Risco": "Cidades dependentes colapsariam e teriam que fechar escolas ou atrasar salários sem o aporte federal."},
    {"Sigla / Termo": "RGPS Rural", "Nome Oficial": "Aposentadorias e Pensões Rurais do Regime Geral", "O que é na prática": "Benefícios previdenciários pagos a agricultores familiares, meeiros e pequenos produtores rurais.", "Por que importa para a cidade": "Em mais de 70% dos pequenos municípios, o volume das aposentadorias rurais supera a arrecadação própria e o FPM.", "Atenção / Risco": "A desvinculação do salário mínimo geraria quebra imediata do pequeno comércio, feiras e farmácias locais."},
    {"Sigla / Termo": "BPC / LOAS", "Nome Oficial": "Benefício de Prestação Continuada da Lei Orgânica da Assistência Social", "O que é na prática": "Transferência de 1 salário mínimo mensal a idosos com 65+ anos e pessoas com deficiência em extrema pobreza.", "Por que importa para a cidade": "É a rede de sobrevivência biológica dos mais vulneráveis, garantindo dignidade e comida na mesa.", "Atenção / Risco": "Qualquer endurecimento de regras ou desvinculação atinge famílias sem nenhuma outra renda."},
    {"Sigla / Termo": "CEAF / Alto Custo", "Nome Oficial": "Componente Especializado da Assistência Farmacêutica", "O que é na prática": "Medicamentos de alto valor fornecidos pelo SUS para doenças crônicas complexas e raras.", "Por que importa para a cidade": "Remédios que custam de R$ 2 mil a mais de R$ 50 mil ao mês, impossíveis de pagar pela família média.", "Atenção / Risco": "Atrasos ou descontinuidade colocam em risco de morte imediato os pacientes em tratamento."},
    {"Sigla / Termo": "MEI", "Nome Oficial": "Microempreendedor Individual (SIMEI)", "O que é na prática": "Pequeno negócio formalizado com CNPJ, tributação simplificada e cobertura previdenciária.", "Por que importa para a cidade": "Base da autonomia econômica de milhões de famílias no comércio e serviços.", "Atenção / Risco": "Rede vital de sustentação econômica comunitária e formalização."},
    {"Sigla / Termo": "Microempresas (ME)", "Nome Oficial": "Microempresas Ativas (Painel Mapa de Empresas)", "O que é na prática": "Pequenas empresas com faturamento anual de até R$ 360 mil, englobando comércio de bairro, padarias, mercadinhos e oficinas.", "Por que importa para a cidade": "Maior gerador de postos de trabalho diretos na malha urbana local.", "Atenção / Risco": "Fortemente vulneráveis à queda da massa salarial e do poder de compra local."},
    {"Sigla / Termo": "Bolsa Família", "Nome Oficial": "Programa Bolsa Família (PBF)", "O que é na prática": "Transferência direta de renda com condicionalidades de vacinação e frequência escolar das crianças.", "Por que importa para a cidade": "Injeta dinheiro vivo diretamente nos mercadinhos, açougues e feiras da cidade.", "Atenção / Risco": "Cortes ou bloqueios geram aumento imediato da insegurança alimentar e recessão no varejo de bairro."},
    {"Sigla / Termo": "ETI", "Nome Oficial": "Escolas em Tempo Integral (Programa de Fomento)", "O que é na prática": "Recursos para ampliação da jornada escolar para pelo menos 7 horas diárias, com 3 a 4 refeições.", "Por que importa para a cidade": "Tira crianças e adolescentes da vulnerabilidade das ruas e garante alimentação saudável.", "Atenção / Risco": "Fim do fomento forçaria prefeituras a retornar ao turno reduzido de 4 horas."},
    {"Sigla / Termo": "PNLD", "Nome Oficial": "Programa Nacional do Livro e do Material Didático", "O que é na prática": "Distribuição 100% gratuita de livros didáticos, literários e pedagógicos a todos os alunos da rede pública.", "Por que importa para a cidade": "Desonera as prefeituras e garante material de alta qualidade padronizado a todas as escolas.", "Atenção / Risco": "Cortes transfeririam um custo financeiro insustentável para famílias e municípios."}
]
df_glossario = pd.DataFrame(glossario_cidadao)

# 8. Metodologia e Fontes
try:
    df_metodologia = pd.read_excel("traducao_municipios_consolidado_user_saved.xlsx", sheet_name="Metodologia_Fontes")
except Exception:
    df_metodologia = pd.DataFrame([{"Eixo": "Geral", "Fonte": "Fontes Governamentais Oficiais"}])

# Exportação do Excel multi-abas
with pd.ExcelWriter(OUTPUT_EXCEL, engine='openpyxl') as writer:
    df_aba_narrativas.to_excel(writer, sheet_name='Narrativas_Consolidadas', index=False)
    df_aba_saude.to_excel(writer, sheet_name='Saude_Medicamentos', index=False)
    df_aba_educ.to_excel(writer, sheet_name='Educacao_Escolas', index=False)
    df_aba_prev.to_excel(writer, sheet_name='Previdencia_Economia', index=False)
    df_aba_trabalho.to_excel(writer, sheet_name='Trabalho_e_Renda', index=False)
    df_dicionario_completo.to_excel(writer, sheet_name='Dicionario_de_Dados', index=False)
    df_glossario.to_excel(writer, sheet_name='Glossario_Cidadao', index=False)
    df_metodologia.to_excel(writer, sheet_name='Metodologia_Fontes', index=False)

# Estilização profissional openpyxl
wb = openpyxl.load_workbook(OUTPUT_EXCEL)
header_font = Font(name='Calibri', size=11, bold=True, color='FFFFFF')
header_fill = PatternFill(start_color='1F4E79', end_color='1F4E79', fill_type='solid')

for ws in wb.worksheets:
    ws.views.sheetView[0].showGridLines = True
    for cell in ws[1]:
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    ws.row_dimensions[1].height = 28
    
    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len = 0
        for cell in col[:15]:
            val_str = str(cell.value or '')
            if len(val_str) > max_len:
                max_len = len(val_str)
        ws.column_dimensions[col_letter].width = min(max(max_len + 3, 12), 65)

wb.save(OUTPUT_EXCEL)
print(f"🎉 Arquivo Excel final gerado com sucesso com 8 abas estilizadas e cod_ibge7/cod_ibge6: {OUTPUT_EXCEL}")


⏳ Gravando planilha Excel estruturada: traducao_municipios_consolidado.xlsx...


🎉 Arquivo Excel final gerado com sucesso com 8 abas estilizadas e cod_ibge7/cod_ibge6: traducao_municipios_consolidado.xlsx


## 🎯 Conclusão e Próximos Passos
O objetivo da **Prioridade 4 (Tradução para o Município)** foi plenamente alcançado:
1. **Identificadores Padrão IBGE:** Todos os dados contêm `cod_ibge7` e `cod_ibge6` como primeiras colunas para cruzamentos universais.
2. **Dados 100% municipalizados:** Todas as 5.571 cidades brasileiras têm seus números individualizados nas 4 frentes: Saúde, Educação, Previdência e Trabalho/Renda (CLT, MEI e Bolsa Família).
3. **Histórias + Números Humanizados:** Quando um dado estiver zerado ou sem registro na base oficial, a informação é apresentada com precisão (*"Não há informações oficiais sobre esse assunto."*), evitando construções estranhas como "0 moradores" ou "R$ 0 mil".
4. **Dicionário de Dados Completo:** Cada coluna de cada aba está documentada em uma aba própria de dicionário com unidade, tipo, fonte e período de referência.
5. **Glossário Simples:** Todos os termos técnicos (Bolsa Família, BPC/LOAS, CLT, MEI, Fundeb, VAAF/VAAT/VAAR, Pé-de-Meia, RGPS, CEAF) foram traduzidos para a linguagem cotidiana.
6. **Exportação Concluída:** O arquivo `traducao_municipios_consolidado.xlsx` está pronto para alimentar campanhas, veículos de imprensa, reuniões de bancada e consultas públicas.
